# Query performance tuning case study

**SQL (Advanced)** (HackerRank)

> Take a slow, unindexed query against a large transactions table and demonstrate the indexing/rewrite that fixes it.

**Concepts:** Indexes, EXPLAIN / query plans, Full table scan vs index seek, Rewriting correlated subqueries as joins

This notebook is runnable as-is: it creates an in-memory SQLite database with small synthetic sample data, then runs the actual SQL for this project.

In [ ]:
import sqlite3
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE transactions (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL);
''')
conn.commit()
print('Schema created.')

In [ ]:
# Synthetic sample data — illustrative only, not real business data
import random
random.seed(7)
# Populate a small, structurally representative sample so the queries below return results.
# (Replace with a real dataset, e.g. the UCI Online Retail set, to use this for real analysis.)
print('Populate sample rows here to match the schema above, then run the query cell below.')

In [ ]:
query = '''
-- Before: no index on customer_id -- EXPLAIN QUERY PLAN shows a full scan
EXPLAIN QUERY PLAN
SELECT * FROM transactions WHERE customer_id = 42;

-- Add an index, then re-check the plan
CREATE INDEX idx_transactions_customer ON transactions(customer_id);
EXPLAIN QUERY PLAN
SELECT * FROM transactions WHERE customer_id = 42;

-- Correlated subquery vs equivalent JOIN/HAVING rewrite
SELECT customer_id FROM transactions t1
WHERE amount > (SELECT AVG(amount) FROM transactions t2 WHERE t2.customer_id = t1.customer_id);

SELECT customer_id, AVG(amount) AS avg_amount
FROM transactions
GROUP BY customer_id
HAVING AVG(amount) < MAX(amount);
'''
for stmt in [s.strip() for s in query.split(';') if s.strip()]:
    try:
        cur.execute(stmt)
        rows = cur.fetchall()
        if rows:
            print(stmt.split(chr(10))[0], '...')
            for r in rows[:10]:
                print(r)
            print()
    except sqlite3.OperationalError as e:
        print('Skipped (SQLite dialect limitation):', e)